In [8]:
# Import necessary libraries
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from factor_analyzer import FactorAnalyzer
import matplotlib.cm as cm
from tabulate import tabulate

# Load the dataset
data = pd.read_csv('chbr_100014_AI_R_Data_mmc1.csv')

# Define variables for comfortableness and capability
com_vars = ['Scom3', 'Scom6', 'Scom7', 'Scom8', 'Scom11', 'Scom12', 'Scom13', 
            'Scom14', 'Scom15', 'Scom16', 'Scom17', 'Scom20', 'Scom21', 'Scom22', 
            'Scom27', 'Scom28', 'Scom29', 'Scom31', 'Scom32', 'Scom33', 'Scom36', 
            'Scom37', 'Scom40']

cap_vars = ['Scap1', 'Scap4', 'Scap6', 'Scap7', 'Scap8', 'Scap10', 'Scap11', 
            'Scap15', 'Scap16', 'Scap19', 'Scap20', 'Scap25', 'Scap26', 'Scap27', 
            'Scap30', 'Scap31', 'Scap32', 'Scap36', 'Scap37', 'Scap38', 'Scap40']

# Check if all variables are in the dataset
missing_com = [var for var in com_vars if var not in data.columns]
missing_cap = [var for var in cap_vars if var not in data.columns]

if missing_com:
    print(f"Warning: Missing comfortableness variables: {missing_com}")
if missing_cap:
    print(f"Warning: Missing capability variables: {missing_cap}")

# Filter to only include available variables
com_vars = [var for var in com_vars if var in data.columns]
cap_vars = [var for var in cap_vars if var in data.columns]

# Function to perform PCA and generate plots
def perform_pca_analysis(data, variables, title_prefix):
    # Extract data for the selected variables
    X = data[variables].dropna()
    
    # Standardize the data
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # PCA for Scree Plot
    pca = PCA()
    pca.fit(X_scaled)
    
    # Scree Plot
    plt.figure(figsize=(10, 6))
    components = np.arange(1, min(len(variables) + 1, 21))  # Limit to 20 components for visibility
    plt.plot(components, pca.explained_variance_ratio_[:len(components)], 'o-', linewidth=2)
    plt.axhline(y=0.1, color='r', linestyle='--')  # Common threshold line
    plt.title(f'{title_prefix} Scree Plot')
    plt.xlabel('Principal Component')
    plt.ylabel('Explained Variance Ratio')
    plt.grid(True)
    plt.savefig(f'{title_prefix.lower().replace(" ", "_")}_scree_plot.png', dpi=300, bbox_inches='tight')
    plt.close()
    
    # Calculate cumulative explained variance
    cum_explained_variance = np.cumsum(pca.explained_variance_ratio_)
    
    # Create variance explained table data
    variance_table = []
    for i, var in enumerate(pca.explained_variance_ratio_[:10], 1):
        variance_table.append([f"PC{i}", f"{var:.4f}", f"{cum_explained_variance[i-1]:.4f}"])
    
    # Print variance explained table
    print(f"\n{title_prefix} - Variance Explained:")
    print(tabulate(variance_table, 
                  headers=["Component", "Explained Variance", "Cumulative Variance"], 
                  tablefmt="grid"))
    
    # Factor Loadings
    # We'll use PCA with 2 components
    pca_2 = PCA(n_components=2)
    pca_2.fit(X_scaled)
    
    # Create a DataFrame for loadings
    loadings = pd.DataFrame(pca_2.components_.T, columns=['Component 1', 'Component 2'], index=variables)
    
    # Print loadings table
    loadings_styled = loadings.copy()
    loadings_styled['Component 1'] = loadings_styled['Component 1'].apply(lambda x: f"{x:.4f}")
    loadings_styled['Component 2'] = loadings_styled['Component 2'].apply(lambda x: f"{x:.4f}")
    
    print(f"\n{title_prefix} - Component Loadings:")
    print(tabulate(loadings_styled.reset_index(), 
                  headers=["Variable", "Component 1 (Big Data/Automation)", "Component 2 (Human Judgment)"], 
                  tablefmt="grid"))